# 1) Load CSVs and Summarize Order Demand
This block reads the three source CSV files and prints item quantities needed per order.


In [1]:
import pandas as pd

itemtypes = pd.read_csv("order_itemtypes.csv", header=None, nrows=6)
quantities = pd.read_csv("order_quantities.csv", header=None, nrows=6)
totes = pd.read_csv("orders_totes.csv", header=None, nrows=6)

order_rows = []
for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes.iat[o, k] if k < totes.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            order_rows.append({"order": o + 1, "item_type": int(it), "quantity": int(qt)})

order_df = pd.DataFrame(order_rows)
order_df

,order,item_type,quantity
0,1,3,3
1,1,1,2
2,2,2,1
3,2,3,3
4,2,4,1
5,3,5,2
6,4,0,3
7,4,5,1
8,5,1,1
9,5,2,1


# 2) Load CSVs and Summarize Tote Contents
This block reads the same CSV files and prints which item types/quantities are in each tote.


In [2]:
import pandas as pd

itemtypes = pd.read_csv("order_itemtypes.csv", header=None, nrows=6)
quantities = pd.read_csv("order_quantities.csv", header=None, nrows=6)
totes = pd.read_csv("orders_totes.csv", header=None, nrows=6)

tote_rows = []
for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes.iat[o, k] if k < totes.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            tote_rows.append({"tote": int(tt), "item_type": int(it), "quantity": int(qt)})

tote_df = pd.DataFrame(tote_rows)
tote_df


,tote,item_type,quantity
0,0,3,3
1,0,1,2
2,14,2,1
3,4,3,3
4,14,4,1
5,7,5,2
6,9,0,3
7,10,5,1
8,11,1,1
9,12,2,1


# 3) Order-Driven Heuristic
This block uses the order-driven heuristic that always prioritizes the order with the smallest remaining work (remaining number of items), and pick totes to finish that order


In [9]:
# =====================================
# FULL SRPT + BEST-MATCH TOTE HEURISTIC
# (Heuristic logic unchanged)
# =====================================

from collections import defaultdict
from copy import deepcopy

# -------------------------------------------------
# STEP 1: Convert DataFrames to nested dictionaries
# -------------------------------------------------

order_work = defaultdict(dict)

for _, row in order_df.iterrows():
    order_id = row["order"]
    item_type = row["item_type"]
    qty = row["quantity"]
    order_work[order_id][item_type] = qty


tote_inventory = defaultdict(dict)

for _, row in tote_df.iterrows():
    tote_id = row["tote"]
    item_type = row["item_type"]
    qty = row["quantity"]
    tote_inventory[tote_id][item_type] = qty


# -------------------------------------------------
# STEP 2: Deep copy
# -------------------------------------------------

remaining_orders = deepcopy(order_work)
available_totes = deepcopy(tote_inventory)

# Track pick order inside each tote
tote_pick_sequence = {}   # {tote_id: [item_type, item_type, ...]}


# -------------------------------------------------
# STEP 3: Sort Orders by SRPT
# -------------------------------------------------

order_sequence = sorted(
    remaining_orders.keys(),
    key=lambda o: sum(remaining_orders[o].values())
)


# -------------------------------------------------
# STEP 4: Greedy Tote Selection (UNCHANGED LOGIC)
# -------------------------------------------------

tote_sequence = []

for order_id in order_sequence:
    
    while sum(remaining_orders[order_id].values()) > 0:
        
        best_tote = None
        best_score = 0
        
        for tote_id, tote_items in available_totes.items():
            
            score = 0
            
            for item_type, qty_needed in remaining_orders[order_id].items():
                if item_type in tote_items:
                    score += min(qty_needed, tote_items[item_type])
            
            if score > best_score:
                best_score = score
                best_tote = tote_id
        
        if best_tote is None or best_score == 0:
            break
        
        tote_sequence.append((order_id, best_tote))
        
        # Ensure tote has pick log
        if best_tote not in tote_pick_sequence:
            tote_pick_sequence[best_tote] = []
        
        # Deduct used quantities (LOGGING ADDED ONLY)
        for item_type in remaining_orders[order_id]:
            
            if item_type in available_totes[best_tote]:
                
                used = min(
                    remaining_orders[order_id][item_type],
                    available_totes[best_tote][item_type]
                )
                
                # Record individual picks
                for _ in range(used):
                    tote_pick_sequence[best_tote].append(item_type)
                
                remaining_orders[order_id][item_type] -= used
                available_totes[best_tote][item_type] -= used
        
        # Remove tote if empty
        if all(qty == 0 for qty in available_totes[best_tote].values()):
            del available_totes[best_tote]


# -------------------------------------------------
# STEP 5: OUTPUTS
# -------------------------------------------------

print("Order fulfillment sequence (SRPT):")
print(order_sequence)

print("\nTote loading sequence (order_id, tote_id):")
print(tote_sequence)

item_cols = ["circle", "pentagon", "trapezoid", "triangle",
             "star", "moon", "heart", "cross"]

print("\nItem pick order inside each tote:")
for tote_id, items in tote_pick_sequence.items():
    
    # Convert item numbers to item names
    item_names = [item_cols[i] for i in items]
    
    print(f"Tote {tote_id}: {item_names}")

Order fulfillment sequence (SRPT):
[3, 5, 6, 4, 1, 2]

Tote loading sequence (order_id, tote_id):
[(3, 7), (5, 0), (5, 14), (6, 1), (4, 9), (4, 10), (1, 0), (1, 11), (2, 4), (2, 14), (2, 12)]

Item pick order inside each tote:
Tote 7: ['moon', 'moon']
Tote 0: ['pentagon', 'triangle', 'triangle', 'triangle', 'pentagon']
Tote 14: ['trapezoid', 'star']
Tote 1: ['pentagon', 'pentagon']
Tote 9: ['circle', 'circle', 'circle']
Tote 10: ['moon']
Tote 11: ['pentagon']
Tote 4: ['triangle', 'triangle', 'triangle']
Tote 12: ['trapezoid']


In [10]:
# =====================================
# TOTAL SYSTEM COMPLETION TIME
# =====================================

from copy import deepcopy

NUM_CONVEYORS = 4
LOOP_TIME = 8      # 2 sec per belt × 4 belts
PUSH_TIME = 1      # 1 sec per item removed

# --- Assign orders to conveyors (1–4 repeating) ---
conveyor_orders = {i: [] for i in range(1, NUM_CONVEYORS + 1)}

for i, order_id in enumerate(order_sequence):
    conv = (i % NUM_CONVEYORS) + 1
    conveyor_orders[conv].append(order_id)

# --- Reset simulation copies ---
remaining_orders_sim = deepcopy(order_work)
tote_inventory_sim = deepcopy(tote_inventory)

# --- Tote entry order (unique, in loading order) ---
tote_entry_order = [t[1] for t in tote_sequence]
seen = set()
tote_entry_order = [x for x in tote_entry_order if not (x in seen or seen.add(x))]

# --- Track finish time per conveyor ---
conveyor_time = {i: 0 for i in range(1, NUM_CONVEYORS + 1)}

# --- Simulate each conveyor independently (parallel world) ---
for conv in range(1, NUM_CONVEYORS + 1):
    
    current_time = 0
    
    for order_id in conveyor_orders[conv]:
        
        while sum(remaining_orders_sim[order_id].values()) > 0:
            
            # One full loop passes
            current_time += LOOP_TIME
            
            for tote_id in tote_entry_order:
                
                if tote_id not in tote_inventory_sim:
                    continue
                
                tote_items = tote_inventory_sim[tote_id]
                removed_this_pass = 0
                
                for item_type in remaining_orders_sim[order_id]:
                    
                    if item_type in tote_items:
                        
                        used = min(
                            remaining_orders_sim[order_id][item_type],
                            tote_items[item_type]
                        )
                        
                        if used > 0:
                            removed_this_pass += used
                            remaining_orders_sim[order_id][item_type] -= used
                            tote_items[item_type] -= used
                
                current_time += removed_this_pass * PUSH_TIME
                
                if all(q == 0 for q in tote_items.values()):
                    del tote_inventory_sim[tote_id]
    
    conveyor_time[conv] = current_time


# --------------------------------------
# TOTAL COMPLETION TIME
# --------------------------------------

total_completion_time = max(conveyor_time.values())

print("TOTAL TIME TO COMPLETE ALL ORDERS:")
print(total_completion_time, "seconds")

TOTAL TIME TO COMPLETE ALL ORDERS:
23 seconds


In [11]:
# =====================================
# STEP 6: Create Conveyor CSV Output
# 4 Conveyors (1,2,3,4 repeating)
# =====================================

import pandas as pd

output_rows = []

num_conveyors = 4

for i, order_id in enumerate(order_sequence):
    
    conv_index = (i % num_conveyors) + 1   # 1,2,3,4,1,2,3,4,...
    
    # Initialize row with zeros
    row_dict = {col: 0 for col in item_cols}
    
    # Fill in item quantities for this order
    for item_type, qty in order_work[order_id].items():
        if item_type < len(item_cols):
            row_dict[item_cols[item_type]] = qty
    
    # Add conveyor number
    row_dict = {"conv_num": conv_index, **row_dict}
    
    output_rows.append(row_dict)

# Create DataFrame
output_df = pd.DataFrame(output_rows)

# Save to CSV
output_df.to_csv("order_driven_heuristic_output.csv", index=False)

print(output_df)

   conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
0         1       0         0          0         0     0     2      0      0
1         2       0         1          1         0     0     0      0      0
2         3       0         2          0         0     0     0      0      0
3         4       3         0          0         0     0     1      0      0
4         1       0         2          0         3     0     0      0      0
5         2       0         0          1         3     1     0      0      0
